# Predicting Coffee Yield from Agroecological and Landscape Features

By Adia Redd and Tiffany Tang

### Abstract

This study aimed to identify which agroecological or landscape feature best predicts mean coffee yield in Brazil, with the initial hypothesis that forest cover would be the most influential predictor. Using 51 environmental and landscape variables, we conducted exploratory analyses including histograms, correlation and principal component analysis to characterize data structure and multicollinearity. Preprocessing steps included label encoding, skewness-based log transformation, feature scaling and dimensionality reduction, which resulted in three dataset variants: a scaled/log-transformed, the same data set that was then PCA- transformed, and a reduced feature data set that was scaled, log transformed and PCA- transformed. 

We evaluated Linear Regression, Random Forest Regression and Support vector Regression using GridSearch CV and assessed performance with  R², error metrics, and diagnostic plots. Predictive performance was modest across all approaches, with the best models achieving test R² values of ~0.25–0.30. PCA and feature reduction generally reduced accuracy, suggesting that important signals were lost. SHAP analyses revealed that regional identifiers and temperature-related variables were the most influential predictors across models, while forest cover did not appear prominently in any model or principal component. Feature importance patterns varied across algorithms, limiting interpretability and model reliability.

Overall, results indicate that the current feature set and models are insufficient for accurately predicting coffee yield or identifying a dominant ecological driver. Future directions include exploring neural networks, gradient boosting, spatial modeling, and alternative dimensionality reduction methods to better capture complex environmental relationships.


### Introduction

Coffee production is a major global agricultural sector and Brazil is its largest producer, supplying roughly one-third of the world’s coffee. However, coffee yields can vary substantially across regions due to differences in climate, landscape structure, and ecological conditions. Prior ecological research has suggested that forest cover and landscape heterogeneity can enhance coffee productivity by improving pollination, stabilizing microclimates, and supporting natural pest control (De Marco & Coelho, 2004; Jha et al., 2014). These findings motivate a more quantitative assessment of how agroecological and landscape variables relate to yield across broader spatial scales.

The dataset used in this study (Silva et al., 2021; Zenodo record 5574892) combines municipal-level coffee yield with 51 environmental, climatic, and landscape features, including forest extent, temperature metrics, moisture indices, crop cover, topography, and spatial identifiers. Using this dataset, our goal is to answer the following research question: Which agroecological or landscape features best predict mean coffee yield across Brazilian farms? Based on previous ecological studies, our hypothesis is that forest cover will be the strongest predictor of yield.

Identifying the key drivers of coffee productivity is an important problem for both ecological and agricultural planning. Accurate prediction models can help farmers and policymakers anticipate yield constraints, evaluate the benefits of forested landscapes, and improve management under climate variability. To address this, we apply multiple machine learning models, Linear Regression, Random Forest Regression, and Support Vector Regression, along with extensive preprocessing, PCA, and SHAP-based interpretability to determine which variables consistently contribute to predictive performance.


### Methods

*Dataset and Research Design*

We used the publicly available agroecological dataset from Silva et al. (2021) (Zenodo record 5574892), which contains mean coffee yield and 51 landscape, climatic, and ecological variables from Brazilian municipalities. Our objective was to build predictive models of mean yield and determine which features contribute most strongly to prediction accuracy, with a specific hypothesis that forest cover would be a major predictor.

*Data Preprocessing*

Three categorical variables were label-encoded. Numerical features were screened for skewness using Pandas’ .skew(); variables with absolute skewness > 1.0 were considered for transformation, excluding binary, near-constant, and negative-valued variables. A natural log(+1) transformation was applied only when it reduced skewness. To avoid data leakage, all datasets were split into training and testing sets (80/20) before scaling or PCA.
Three preprocessing pipelines were evaluated:
Log-Transformed + Scaled Dataset
Log-Transformed + Scaled + PCA Dataset
Correlation/Variance-Reduced + Log-Transformed + PCA Dataset
 For Pipeline 3, features with correlation > 0.9 or variance < 0.01 were removed prior to PCA.

*Model Training*

We trained three families of regression models on each dataset variant: Ordinary Linear Regression; regularized models (Ridge, Lasso, ElasticNet); Random Forest Regression (RFR); and Support Vector Regression (SVR). Hyperparameters for all non-ordinary models were optimized using GridSearchCV with five-fold cross-validation. For RFR, the tree-splitting criterion and the maximum number of features considered at each split were tuned, with random_state=8 and out-of-bag error estimation enabled. For the SVR model, we performed a grid search over the C, gamma, and epsilon hyperparameters to identify the best-performing configuration of the RBF kernel. The RBF kernel was selected because it can model non linear relationships, which are expected in ecological yield data. 

*Model Evaluation*

Performance was assessed using R², mean absolute error (MAE), and root mean squared error (RMSE), which are standard metrics for regression prediction. Residual plots and QQ plots were generated to evaluate model assumptions and residual behavior. Feature importance and interpretability were assessed using SHAP values for each best-performing model on each dataset variant.

*Reproducibility*

All analyses were conducted in Python. Full preprocessing, modeling and visualization code is included below to ensure complete reproducibility. 



### Results

All three of the model families were unable to reliably predict the mean yield. When identifying the optimal model using GridSearchCV, all the best models performed poorly in cross validation reporting R² scores between -0.037 and 0.30. When tasked with predicting yield using the test split, the best models achieving test R² values of ~0.25–0.30.

*Linear Regression*
The linear regression models exhibited signs of overfitting...

*Support Vector Regression*

*Random Forest Regression*

Random forest regression produced the best performing model of all three families. The best model, trained on the data that was simply log-transformed and scaled (dataset variant 1), defined the loss function as squared error and the maximum number of features to be considered when making the split in a given tree equal to the log base 2 of the number of samples. The R² score of this model with the test set is 0.28. GridSearch reported the same optimal model when using the remaining two dataset variants for fitting, where the loss function is absolute error and the maximum number of features to be considered when making the split equal to the square root of the number of samples. This model returned R² scores of 0.28 and -0.05 when tasked with predicting the mean yield using their respective test sets.
Despite this poor correlation between the predicted and actual y-values, the residuals plots did not indicate any extremely abnormal behavior for either model. The residuals produced by the top scoring model did, however, display a very mild positive correlation between the actual and predicted mean yield. This could indicate that the dataset could benefit from a more complex model to explain the relationship between the features and the mean yield since there may be a violation of the assumptions associated with regression. The remaining two models produced residual plots with no discernible pattern. To confirm the normality of the errors, the residuals of each model were plotted on a QQ-plot, where it became apparent thar none the residuals were not normally distributed, despite what the actual residual plots themselves may have suggested.

SHAP summaries were generated for both models to investigate which feature was the most important in the model's decision to predict a value for a given sample. The first model's SHAP summary reported the top five most important features to all be temperature-related, aside from argotox_P, which is the percentage of farms in the municipality that used the ArgoTox plant protection agent. For all features there is no clear separation of high and low values in terms of SHAP value. This indicates that for most samples, the model had a difficult time partitioning a sample into one node or the other when a split in a tree arose because it could not be determined if a high or low value was important or not. When predicting y-values for the second dataset variant's test split, the second model selected PC2 as it's most important feature and PC1 as it's least important. When predicting the y-values for the third dataset variant's test split, PC1 was selected as most important and PC7 as least important. Despite being the same model, the important components changes drastically. Since PC2 was indicated a number of times across the models, it will be discussed separately. PC1 is composed mostly of geographic features like Euclidean distance between forest fragments and coffee fields, longitude/latitude, and municipality identifiers.

*Principal Component 2*

Since PC2 was repeatedly revealed as an important feature we investigated which features were used to create it. In the full feature PCA, PC2 is dominated by temperature-related variables, along with average pollination activity and latitude, which all load positively. Elevation, warm-season average temp, and temperature range load negatively. After removing correlated features, the structure of PC2 shifts: elevation becomes the strongest driver, loading positively, and is accompanied by warm season precipitation-related variables, while several cool-season precipitation-related variables load negatively,
Both PC2’s contain temperature features, but neither load forest cover into their composition.

### Conclusion

In conclusion, the models were not strong enough to predict reliably predict mean yield, and thus unable to indicate the most important feature for making the predictions. All models showed modest predictive performance, with test R² around 0.25–0.30 at best. The errors produced by the SVR and RFR models appeared to be normally distributed, but were revealed not to be by the QQ-plot, indicating the model may not be able to sort out the complexity of the dataset. While the errors produced by the linear regression models were normally distributed, indicating an appropriate model, but it was not a good model as indicated by the R² scores, which were the lowest of all three model families. Some features did appear to be important based on the SHAP summaries, like codigo_ibg, many of the temperature related features, and PC2, but this was not consistent across models. Forest cover did not appear at all as an important feature from dataset variant 1, and was not apart of the composition of any relevant principal components used in the other two dataset variants. There is not sufficient evidence to reject or accept the proposed hypothesis.

# CODE

In [ ]:
# import libraries
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import tqdm as notebook_tqdm
import shap
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import statsmodels.api as sm

In [ ]:
# read in the data and take a look at it
# change path to fit your needs
file_path = r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\cleaned_coffee.csv"

#file_path = r'/Users/adia/PyCharmMiscProject/ML/Final/cleaned_coffee.csv'

# Read the CSV
df = pd.read_csv(file_path, sep = ';') # semicolon delimited file

# Take a quick look at the data
print(df.head())

## Exploratory Analysis

In [ ]:
# Label Encoding
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
categorical_cols
df_encoded = df.copy() # copy the dataframe

label = LabelEncoder() # label encode each column that is non numeric
for col in categorical_cols:
    df_encoded[col] = label.fit_transform(df_encoded[col].astype(str))

df_encoded = df_encoded.drop(columns=["ID"]) # don't need the ID column
# save this
df_encoded.to_csv("df_encoded.csv", index=False)

In [ ]:
# Exploratory analysis
# Correlation
corr = df_encoded.corr()
corr.head() # just to take a look
plt.figure(figsize=(14, 12))# set figure size
sns.heatmap(corr, cmap= "coolwarm", annot= False)
plt.title("Correlation Heatmap of the Label Encoded Data")
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\correlation_heatmap.png")
plt.show()

# Feature histogram
df_encoded.hist(figsize= (30,30))
plt.title("Feature Distributions")
#plt.savefig(r'/Users/adia/PyCharmMiscProject/ML/Final/Figures/feature_dist_histograms.png')
plt.show()

In [ ]:
# Log transform skewed features
# function to detect skewness- log transformations cannot change left skew!
def detect_skewed_features(df, skew_threshold=1.0):
    skewed_features = []

    for col in df.columns:
        series = df[col]
        # Skip binary or near-constant
        if series.nunique() <= 3:
            continue
        # Skip negative values (log1p cannot handle)
        if (series < 0).any():
            continue
        # Original skewness
        orig_skew = series.skew()
        # Must be skewed enough to consider transforming
        if abs(orig_skew) <= skew_threshold:
            continue
        # Skewness after log1p
        log_skew = np.log1p(series).skew()
        # Only transform if log reduces skewness
        if abs(log_skew) < abs(orig_skew):
            skewed_features.append(col)

    return skewed_features
# let's see the skewed columns
skewed_cols = detect_skewed_features(df_encoded)

# just in case keep the non-skewed columns separately
non_skewed_cols = [c for c in df_encoded.columns if c not in skewed_cols]

print("Skewed columns to log-transform:")
print(skewed_cols)
len(skewed_cols)

# apply log1p to skewed features
df_log = df_encoded.copy() # copy the dataset over

df_log[skewed_cols] = np.log1p(df_log[skewed_cols])

# define X and y, we will be using these from here on forward!!!!
y = df_log["yield.mean"]
X = df_log.drop(columns=["yield.mean"])

# lets replot this histogram to see if the feature skew is better
df_log.iloc[:,:52].hist(figsize= (30,30))
plt.title("Feature Distributions After Log Transformation")
plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\feature_log_hist.png")
plt.show()

In [ ]:
# Exploratory Analysis
# PCA, using the encoded and log transformed dataset 
# first scale, only want to look at the predictors
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# save as a dataframe
pca_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_columns)

# Add the target column back
# This is for our visualization plots, it was not used to compute the PC's
df_pca["yield.mean"] = y.values

df_pca.plot(kind='scatter', x='PC1', y='PC2', c= df_pca["yield.mean"],  cmap='viridis')
plt.title('Scatter Plot of Data Using PC1/PC2')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True)
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_scatter.png")
plt.show()

# 3D plot
from mpl_toolkits.mplot3d import Axes3D

# assign the definitions of each quartile
lowbound = df_pca['yield.mean'].min()
q1 = df_pca['yield.mean'].quantile(.25)
q2 = df_pca['yield.mean'].quantile(.5)
q3 = df_pca['yield.mean'].quantile(.75)
highbound = df_pca['yield.mean'].max()

# separate by quartile function
def assign_quartile(value):
    if value <= q1:
        return 0
    elif value <= q2:
        return 1
    elif value <= q3:
        return 2
    else:
        return 3

colors = df_pca['yield.mean'].apply(assign_quartile) # color code the quartiles

# plot

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
scatplt = ax.scatter(df_pca['PC1'], df_pca['PC2'], df_pca['PC3'], c=colors, cmap='viridis',s=5)

cbar = plt.colorbar(scatplt, ax=ax, ticks=[0, 1, 2, 3],shrink=0.4, pad=.1)
cbar.set_label('Yield Quartile')
cbar.set_ticklabels(['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

ax.set_xlabel('PC1')
ax.set_ylabel('PC2',labelpad=10)
ax.set_zlabel('PC3')
ax.set_title('Samples colored by Yield Mean Quartiles on the top 3 PC Axes')

#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_scatter3D.png")
plt.show()

# see if any of the PC's have a correlation with the yield mean
# Compute correlation between each PC and the target
pc_target_corr = df_pca.drop("yield.mean", axis=1).corrwith(df_pca["yield.mean"])
pc_target_corr
# output was not promising

# total variance plots
plt.figure(figsize=(10,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o') # plot the variance ratio
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_cumulative_var.png")
plt.show()

## Data Processing

In [ ]:
# Prepare out datasets to be used
# Split and scale
# train, test split - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, 
    random_state= 32 # this is for reproducibility purposes
)
# no validation set because we will include this in the grid search

# scale the split data
scaler = StandardScaler()

# ******FIRST DATASET******
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# PCA dataset creation
# scale first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # scale on the unscaled dataset
X_test_scaled  = scaler.transform(X_test)

# PCA transformation to the scaled dataset
pca = PCA()  
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# make them into dataframes
pca_cols = [f"PC{i+1}" for i in range(X_train_pca.shape[1])]

X_train_pca = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train.index)
X_test_pca  = pd.DataFrame(X_test_pca, columns=pca_cols, index=X_test.index)

# *****SECOND DATASET*****
# running on the first 7 PC's- since they explain about 75% of the variance
X_train_pca_selected = X_train_pca.iloc[:, :7]
X_test_pca_selected  = X_test_pca.iloc[:, :7]


# Third dataset creation
# remove 0 variance features and highly correlated features
from sklearn.feature_selection import VarianceThreshold

# set the threshold to 0.01, if its less than that then we remove
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)

# get list of retained column names
cols_vt = X.columns[vt.get_support()]
X_vt_df = pd.DataFrame(X_vt, columns=cols_vt)

# remove the highly correlated features
corr = X_vt_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
print(to_drop)

X_reduced = X_vt_df.drop(columns=to_drop)

# split scale run PCA on this reduced features
X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_red_scaled = scaler.fit_transform(X_train_red)
X_test_Red_scaled  = scaler.transform(X_test_red)

pca = PCA()
X_train_pca_red = pca.fit_transform(X_train_red_scaled)
X_test_pca_red  = pca.transform(X_test_Red_scaled)

# *****THIRD DATASET*****
# run on the first 7 PC's like before
X_train_red_selected = X_train_pca_red[:, :7]
X_test_red_selected  = X_test_pca_red[:, :7]

# Convert PCA arrays to DataFrames so SHAP can use .sample()
pc_names = [f"PC{i+1}" for i in range(7)]

X_train_red_selected = pd.DataFrame(X_train_red_selected, columns=pc_names)
X_test_red_selected  = pd.DataFrame(X_test_red_selected, columns=pc_names)


In [ ]:
# see what features contribute to PC2
# Get PCA loadings (components)
loadings = pd.DataFrame(
    pca.components_,
    columns=X_train.columns,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Sort PC2 features by absolute contribution, but keep actual signed loadings
pc2_sorted = loadings.loc["PC2"].reindex(
    loadings.loc["PC2"].abs().sort_values(ascending=False).index
)

print(pc2_sorted.head(10))  # top 10 features
# PCA loadings
loadings = pd.DataFrame(
    pca.components_,
    columns=X_train.columns,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Sort by absolute loading values (strongest contributors)
pc2_sorted = loadings.loc["PC2"].sort_values(key=lambda x: np.abs(x), ascending=False)

# Take top 10 contributors
top_pc2 = pc2_sorted.head(10)

# Plot
plt.figure(figsize=(8, 6))
top_pc2[::-1].plot(kind="barh")   # reverse for descending order visually
plt.title("Top Feature Contributions to PC2")
plt.xlabel("Loading Weight")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# See what features contribute to PC2 for the Reduced dataset
# get loadings
loadings_red = pd.DataFrame(
    pca.components_,
    columns=X_train_red.columns,   # original reduced feature names
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# print
pc2_loadings = loadings_red.loc["PC2"].sort_values(key=lambda x: abs(x), ascending=False)
print("PC2 Loadings (Reduced-Feature PCA):")
print(pc2_loadings.head(15))   # top 15 contributors

# plot
plt.figure(figsize=(8, 6))
pc2_loadings.head(15).plot(kind='barh')
plt.xlabel("Loading Weight")
plt.title("Top PC2 Loadings (Reduced-Feature PCA)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Support Vector Regression

In [ ]:
# Support Vector Regression Code
# scale the data and ft the model
# ***** First Dataset*****
svr_model = Pipeline([ # use pipleline to scale and train the data
    ("scaler", StandardScaler()), # scale after splitting to prevent data leakage!
    ("svr", SVR(kernel= "rbf")) # chose RBF kernel since the relationship between X and y is most likely non linear
])
#hyperparameter grid search for the best parameters
param_grid = {
    "svr__C": [0.1, 1, 10, 100],
    "svr__gamma": ["scale", "auto", 0.01, 0.001, 0.0001],
    "svr__epsilon": [0.1, 0.5, 1.0]
}

grid = GridSearchCV(
    estimator= svr_model,
    param_grid= param_grid,
    cv=5, # cross validation
    scoring="r2",
    n_jobs=-1, 
    verbose= 2
)

# fit this grid only on the training data
grid.fit(X_train, y_train) # since we are scaling in the pipeline we won't use the scaled data

print("Best R² training:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_model = grid.best_estimator_ # assign out best

# evaluate on the test set
y_pred = grid.predict(X_test)

print("Test R²:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# SHAP figure code
explainer = shap.KernelExplainer(grid.predict, X_train.sample(50))
shap_values = explainer.shap_values(X_test.sample(50))

shap.summary_plot(shap_values, X_test.sample(50))

#residuals Code
import statsmodels as sm
residuals = y_test - y_pred

# residuals
sns.residplot(x=y_pred, y=residuals, lowess=True)
plt.xlabel("Predicted Mean Yield")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (Log-Transformed & Scaled Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_residual_plot.png", dpi=300)
plt.show()

# qq
sm.qqplot(residuals, line='45')
plt.title("SVR Residuals QQ Plot (Log-Transformed & Scaled Data)")
plt.tight_layout()
plt.savefig("svr_qq_plot.png", dpi=300)
plt.show()

# *****Second Dataset*****
# fit the model
svr_model_pca = SVR(kernel="rbf")

#hyper parameters
param_grid_pca = {
    "C": [1, 10, 100, 1000, 10000],
    "gamma": [1, 0.5, 0.1, 0.01, 0.001],
    "epsilon": [0.001, 0.01, 0.1]
}

# grid search
grid = GridSearchCV(
    estimator=svr_model_pca, 
    param_grid= param_grid_pca, 
    cv= 5, 
    scoring = "r2", 
    n_jobs=1, 
)
# fit this grid only on the training data
grid.fit(X_train_pca_selected, y_train)
# PCA is only for the predictors

print("Best R²:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_pca = grid.best_estimator_

# evaluate on the test set
y_pred_pca = grid.predict(X_test_pca_selected)

print("Test R²:", r2_score(y_test, y_pred_pca))
print("Test MAE:", mean_absolute_error(y_test, y_pred_pca))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_pca)))

# shap values on the PCA set: 
explainer = shap.KernelExplainer(grid.predict, X_train_pca_selected.sample(50))
shap_values = explainer.shap_values(X_test_pca_selected.sample(50))

shap.summary_plot(shap_values, X_test_pca_selected.sample(50))
# residuals for PCA model
residuals_pca = y_test - y_pred_pca

# residual
sns.residplot(x=y_pred_pca, y=residuals_pca, lowess=True)
plt.xlabel("Predicted Mean Yield (PCA SVR)")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (PCA Transformed Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_pca_residual_plot.png", dpi=300)
plt.show()

# qq
sm.qqplot(residuals_pca, line='45')
plt.title("SVR Residuals QQ Plot (PCA Transformed Data)")
plt.tight_layout()
plt.savefig("svr_pca_residual_qq_plot.png", dpi=300)
plt.show()

# *****Third Dataset*****
# fit the model
svr_model_pca_red = SVR(kernel="rbf")

#hyper parameters
param_grid_pca_red = {
    "C": [10, 100, 1000, 10000], 
    "gamma": [ 1, 0.1, 0.01, 0.001], 
    "epsilon": [0.01, 0.05, 0.1]
}

# grid search
grid = GridSearchCV(
    estimator=svr_model_pca_red, 
    param_grid= param_grid_pca_red, 
    cv= 5, 
    scoring = "r2", 
    n_jobs=1, 
)
# fit this grid only on the training data
grid.fit(X_train_red_selected, y_train)
# PCA is only for the predictors

print("Best R²:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_pca = grid.best_estimator_
# evaluate on the test set
y_pred_red = grid.predict(X_test_red_selected)

print("Test R²:", r2_score(y_test, y_pred_red))
print("Test MAE:", mean_absolute_error(y_test, y_pred_red))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_red)))

#Residuals
# Calculate residuals
residuals_red = y_test - y_pred_red

# residual
sns.residplot(x=y_pred_red, y=residuals_red, lowess=True)
plt.xlabel("Predicted Mean Yield (PCA Reduced SVR)")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (Reduced and PCA Transformed Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_pca_residual_plot.png", dpi=300)
plt.show()

# qq
sm.qqplot(residuals_red, line='45')
plt.title("SVR Residuals QQ Plot (Reduced and PCA Transformed Data)")
plt.tight_layout()
plt.savefig("svr_pca_residual_qq_plot.png", dpi=300)
plt.show()

# shap on the features
explainer = shap.KernelExplainer(grid.predict, X_train_red_selected.sample(50))
shap_values = explainer.shap_values(X_test_red_selected.sample(50))

shap.summary_plot(shap_values, X_test_red_selected.sample(50))


## Random Forest Regression

In [ ]:
    # Defining the fixed parameters of RFR model
rfr_model = RandomForestRegressor(random_state= 8, oob_score = True)

    # Parameter grid to be tested in GridSearchCV
rfr_param_grid = { 'criterion':['squared_error', 'absolute_error', 'friedman_mse'],
                   'max_features' : [None, 'sqrt', 'log2'] }

parameter_tests = GridSearchCV(
    estimator=rfr_model,
    param_grid=rfr_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2)

In [ ]:
# *** FIRST DATASET VARIANT ***
parameter_tests.fit(X_train_scaled, y_train)

    # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
optimized_rfr_model = parameter_tests.best_estimator_
rfr_y_pred = optimized_rfr_model.predict(X_test_scaled)

    # model performance
print("Test R²:", r2_score(y_test, rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, rfr_y_pred)))

    # creating SHAP summary
rfr_explainer = shap.TreeExplainer(optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_scaled)
shap.summary_plot(rfr_shap_values, X_test_scaled, feature_names=X.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# *** SECOND DATASET VARIANT***
parameter_tests.fit(X_train_pca_selected, y_train)

    # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
pca_optimized_rfr_model = parameter_tests.best_estimator_
pca_rfr_y_pred = pca_optimized_rfr_model.predict(X_test_pca_selected)

    # model performance
print("Test R²:", r2_score(y_test, pca_rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, pca_rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pca_rfr_y_pred)))

    # generating SHAP summary
rfr_explainer = shap.TreeExplainer(pca_optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_pca_selected)
shap.summary_plot(rfr_shap_values, X_test_pca_selected, feature_names=X_train_pca_selected.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_LPS_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# *** THIRD DATASET VARIANT***
parameter_tests.fit(X_train_red_selected, y_train)

   # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
red_pca_optimized_rfr_model = parameter_tests.best_estimator_
pca_red_rfr_y_pred = red_pca_optimized_rfr_model.predict(X_test_red_selected)

    # model performance
print("Test R²:", r2_score(y_test, pca_red_rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, pca_red_rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pca_red_rfr_y_pred)))

    # generating SHAP summary
rfr_explainer = shap.TreeExplainer(red_pca_optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_red_selected)
shap.summary_plot(rfr_shap_values, X_test_red_selected, feature_names=X_train_red_selected.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_LPS_red_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# generating residuals plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(x=rfr_y_pred, y=y_test, color = "olive", ax = axes[0])
axes[0].set_xlabel("Predicted Mean Yield")
axes[0].set_ylabel("Residuals")
axes[0].set_title('Log Trans. and Scaled RFR Residuals')
axes[0].grid(True)

sns.residplot(x=pca_rfr_y_pred, y=y_test, color = 'magenta', ax = axes[1])
axes[1].set_xlabel("Predicted Mean Yield")
axes[1].set_ylabel("Residuals")
axes[1].set_title(' Scaled, Log + PCA Trans RFR Residuals')
axes[1].grid(True)

sns.residplot(x=pca_red_rfr_y_pred, y=y_test,color = 'blue', ax = axes[2])
axes[2].set_xlabel("Predicted Mean Yield")
axes[2].set_ylabel("Residuals")
axes[2].set_title(' Scaled, Log + PCA Reduced RFR Residuals')
axes[2].grid(True)

#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_resid_multi.png")
plt.show()

In [ ]:
# generating QQ-plots
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sm.qqplot(data=(y_test - rfr_y_pred), line='45',ax = axes[0])
axes[0].set_title('Log Trans. and Scaled Residuals QQ Plot')

sm.qqplot(data=(y_test - pca_rfr_y_pred), line='45', ax = axes[1])
axes[1].set_title('Scaled, Log + PCA Trans Residuals QQ Plot')

sm.qqplot(data=(y_test - pca_red_rfr_y_pred), line='45', ax = axes[2])
axes[2].set_title('Scaled, Log + PCA Reduced Residuals QQ Plot')

#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_qq_multi.png")
plt.show()